# Precio contra avalúo catastral

In [1]:
from eda_utils import *   # carga y prepara la base (capítulo 1)

El avalúo de la base es una sola foto del catastro: el 94% de los predios vendidos en
años distintos tiene el mismo avalúo en todas sus ventas. Por eso llevamos el precio a
pesos de 2022 (el año de esa foto) antes de dividirlo por el avalúo. Una razón de 1
significa que el catastro coincide con el mercado.

In [2]:
# El avalúo de la base es una sola foto del catastro: el mismo predio vendido en años
# distintos tiene el mismo avalúo. Por eso el precio se lleva a pesos de 2022 antes de
# dividir; si no, las ventas recientes parecerían más "subvaloradas" solo por la inflación.
misma_foto = (df.dropna(subset=["avaluo", "codigo_predial"]).groupby("codigo_predial")
                .agg(anios=("anio", "nunique"), avaluos=("avaluo", "nunique")).query("anios > 1"))
print(f"Predios vendidos en años distintos: {len(misma_foto):,}; con el mismo avalúo en todas sus ventas: "
      f"{(misma_foto.avaluos == 1).mean():.0%}")

a_2022 = ipc_anio.loc[2022] / ipc_anio
brecha = viv[viv.avaluo.notna()].copy()
brecha["razon"] = brecha.precio * brecha.anio.map(a_2022) / brecha.avaluo
brecha = brecha[brecha.razon.between(0.1, 20)]
r = brecha.razon

fig = make_subplots(1, 2, subplot_titles=("Razón precio / avalúo", "Por estrato"))
fig.add_histogram(x=r.clip(upper=6), nbinsx=70, marker_color=AZUL, row=1, col=1,
                  hovertemplate="%{x:.1f} veces: %{y} ventas<extra></extra>")
fig.add_vline(x=1, line_dash="dash", line_color="black", row=1, col=1, annotation_text="precio = avalúo")
for nombre, e, color in zip(NOMBRES_ESTRATO, ESTRATOS, COLORES_ESTRATO):
    fig.add_box(y=brecha.loc[brecha.estrato == e, "razon"], name=nombre.replace("Estrato ", ""),
                boxpoints=False, marker_color=color, row=1, col=2)
fig.add_hline(y=1, line_dash="dash", line_color="black", row=1, col=2)
fig.update_yaxes(range=[0, 4], row=1, col=2)
fig.update_xaxes(title_text="veces el avalúo (precio en pesos de 2022)", row=1, col=1)
fig.update_xaxes(title_text="estrato", row=1, col=2)
fig.update_layout(showlegend=False)
mostrar(fig, "¿Cuánto se paga por encima del avalúo catastral?")

print(f"Ventas con avalúo: {len(brecha):,}   mediana: {r.median():.2f}   "
      f"por encima del avalúo: {(r > 1).mean():.0%}   al doble o más: {(r >= 2).mean():.0%}")
W, p = stats.wilcoxon(np.log(r), alternative="greater")
anotar("¿Se paga por encima del avalúo?", "Wilcoxon de rangos con signo", W, p, f"mediana = {r.median():.2f}")
H, p, eps2 = kruskal_por_grupo(brecha, "razon", "estrato", ESTRATOS)
anotar("¿La razón precio/avalúo cambia con el estrato?", "Kruskal-Wallis", H, p, f"épsilon² = {eps2:.2f}")
d = brecha.dropna(subset=["estrato_num"])
rho, p = stats.spearmanr(d.estrato_num, d.razon)
anotar("¿A mayor estrato, mayor subvaloración?", "Correlación de Spearman", rho, p, f"rho = {rho:.2f}")
brecha.groupby("estrato_txt").razon.median().round(2).rename("razón mediana").to_frame().T

Predios vendidos en años distintos: 1,007; con el mismo avalúo en todas sus ventas: 94%


Ventas con avalúo: 11,955   mediana: 1.54   por encima del avalúo: 80%   al doble o más: 26%
Wilcoxon de rangos con signo: estadístico = 58,529,814.000   p = < 0,001   mediana = 1.54
Kruskal-Wallis: estadístico = 139.593   p = < 0,001   épsilon² = 0.01
Correlación de Spearman: estadístico = 0.084   p = < 0,001   rho = 0.08


estrato_txt,Estrato 1,Estrato 2,Estrato 3,Estrato 4,Estrato 5,Estrato 6,Sin estrato
razón mediana,1.38,1.30,1.55,1.65,1.63,1.66,1.04


In [3]:
por_loc = (brecha[brecha.localidad.notna() & ~brecha.localidad.isin(NO_BARRIOS)]
           .groupby("localidad").razon.median().sort_values())
por_bar = (brecha[brecha.barrio.notna() & ~brecha.barrio.isin(NO_BARRIOS)]
           .groupby("barrio").razon.agg(["median", "size"]).query("size >= 30").sort_values("median"))

fig = make_subplots(1, 2, subplot_titles=("Por localidad", "Barrios con 30 ventas o más (extremos)"),
                    horizontal_spacing=0.25)
fig.add_bar(x=por_loc.values, y=por_loc.index.str.title(), orientation="h", marker_color=AZUL, row=1, col=1,
            hovertemplate="%{y}: %{x:.2f} veces<extra></extra>")
extremos = pd.concat([por_bar.head(6), por_bar.tail(6)])
fig.add_bar(x=extremos["median"], y=extremos.index.str.title(), orientation="h", row=1, col=2,
            marker_color=[AZUL] * 6 + [ROJO] * 6, hovertemplate="%{y}: %{x:.2f} veces<extra></extra>")
for c in (1, 2):
    fig.add_vline(x=1, line_dash="dash", line_color="black", row=1, col=c)
fig.update_xaxes(title_text="razón mediana")
fig.update_layout(showlegend=False)
mostrar(fig, "La brecha con el avalúo según la zona", alto=460)
print(f"Localidades: de {por_loc.min():.2f} ({por_loc.index[0].title()}) a {por_loc.max():.2f} ({por_loc.index[-1].title()})")
print(f"Barrios: de {por_bar['median'].min():.2f} ({por_bar.index[0].title()}) a "
      f"{por_bar['median'].max():.2f} ({por_bar.index[-1].title()})")

Localidades: de 1.25 (Suroriente) a 1.68 (Riomar)
Barrios: de 0.50 (Altos Del Limon) a 2.06 (La Magdalena)


La vivienda típica se vende a 1,54 veces su avalúo, el 80% por encima de él y una de
cada cuatro al doble o más. La brecha es menor en los estratos 1 y 2 (1,3-1,4) que en
los 4 a 6 (1,6-1,7), pero el estrato explica muy poco de ella (épsilon² = 0,01). Donde
más cambia es entre zonas: de 1,25 en Suroriente a 1,68 en Riomar, y entre barrios de
0,5 a 2,1. El catastro se aleja más del mercado en unos barrios que en otros, no tanto
según el estrato.

Como el avalúo es de un solo año, no analizamos cómo cambia la brecha en el tiempo.